In [1]:
# ==========================================
# 📊 ANALISIS KEMAMPUAN SISWA + SARAN PERBAIKAN
# ==========================================

import pandas as pd
import re
from google.colab import files

# --- 1️⃣ Upload dua file Excel ---
print("📂 Silakan upload file hasil evaluasi siswa (contoh: hasil_evaluasi.xlsx)")
uploaded_siswa = files.upload()

print("\n📂 Silakan upload file soal dan indikator (contoh: soal_dan_indikator.xlsx)")
uploaded_indikator = files.upload()

# --- 2️⃣ Baca file Excel ---
nama_file_siswa = list(uploaded_siswa.keys())[0]
nama_file_indikator = list(uploaded_indikator.keys())[0]

df_siswa = pd.read_excel(nama_file_siswa)
df_indikator = pd.read_excel(nama_file_indikator)

# --- 3️⃣ Bersihkan nama kolom ---
df_siswa.columns = df_siswa.columns.str.strip()
df_indikator.columns = df_indikator.columns.str.strip()

# --- 4️⃣ Ubah data siswa ke format long ---
soal_cols = [col for col in df_siswa.columns if 'Soal' in col]
df_long = df_siswa.melt(id_vars=['Nama', 'Absen', 'Nilai'],
                        value_vars=soal_cols,
                        var_name='Soal', value_name='Jawaban')

# --- 5️⃣ Ambil nomor soal dari nama kolom ---
df_long['Nomor Soal'] = df_long['Soal'].apply(
    lambda x: int(re.findall(r'\d+', str(x))[0]) if re.findall(r'\d+', str(x)) else None
)

# --- 6️⃣ Samakan nama kolom indikator dan ambil yang penting ---
if 'No' in df_indikator.columns:
    df_indikator.rename(columns={'No': 'Nomor Soal'}, inplace=True)

if 'Indikator' not in df_indikator.columns:
    raise KeyError("❌ Kolom 'Indikator' tidak ditemukan di file soal. Pastikan nama kolomnya benar.")

df_indikator = df_indikator[['Nomor Soal', 'Indikator']]

# --- 7️⃣ Gabungkan data siswa dengan indikator ---
df_long = pd.merge(df_long, df_indikator, on='Nomor Soal', how='left')

# --- 8️⃣ Konversi jawaban Benar/Salah ke angka ---
df_long['Skor'] = df_long['Jawaban'].apply(lambda x: 1 if str(x).strip().lower() == 'benar' else 0)

# --- 9️⃣ Hitung rata-rata skor per indikator per siswa ---
pivot = df_long.groupby(['Nama', 'Indikator'])['Skor'].mean().reset_index()

# --- 🔟 Analisis kemampuan siswa + saran perbaikan ---
def analisis_siswa_dengan_saran(pivot_df):
    hasil = []
    for siswa in pivot_df['Nama'].unique():
        data = pivot_df[pivot_df['Nama'] == siswa]
        kelebihan = data[data['Skor'] >= 0.75]['Indikator'].tolist()
        kekurangan = data[data['Skor'] < 0.75]['Indikator'].tolist()

        # Format bullet-list (per baris)
        kelebihan_text = '\n'.join([f"• {k}" for k in kelebihan]) if kelebihan else '• Belum tampak dominan'
        kekurangan_text = '\n'.join([f"• {k}" for k in kekurangan]) if kekurangan else '• Tidak ada kekurangan yang menonjol'

        # Saran perbaikan otomatis
        if kekurangan:
            saran_list = [f"• Perbanyak latihan pada indikator: {k.lower()}." for k in kekurangan]
            saran_text = '\n'.join(saran_list)
        else:
            saran_text = '• Pertahankan hasil belajar yang sudah baik dan bantu teman lain memahami materi.'

        hasil.append({
            'Nama': siswa,
            'Kelebihan': kelebihan_text,
            'Kekurangan': kekurangan_text,
            'Saran Perbaikan': saran_text
        })
    return pd.DataFrame(hasil)

analisis_df = analisis_siswa_dengan_saran(pivot)

# --- 11️⃣ Tampilkan hasil akhir dengan teks rata kiri ---
print("\n✅ HASIL ANALISIS KEMAMPUAN SISWA + SARAN PERBAIKAN ✅")

display(
    analisis_df.style
    .set_properties(**{
        'white-space': 'pre-wrap',   # agar baris baru (\n) terbaca
        'text-align': 'left',        # teks rata kiri
        'vertical-align': 'top'      # teks di atas sel
    })
)

# --- 12️⃣ Simpan hasil ke Excel ---
analisis_df.to_excel('hasil_analisis_dengan_saran.xlsx', index=False)
print("\n📁 File 'hasil_analisis_dengan_saran.xlsx' telah dibuat dan siap diunduh.")


📂 Silakan upload file hasil evaluasi siswa (contoh: hasil_evaluasi.xlsx)


Saving hasil_evaluasi(1).xlsx to hasil_evaluasi(1).xlsx

📂 Silakan upload file soal dan indikator (contoh: soal_dan_indikator.xlsx)


Saving soal_dan_indikator.xlsx to soal_dan_indikator.xlsx

✅ HASIL ANALISIS KEMAMPUAN SISWA + SARAN PERBAIKAN ✅


,Nama,Kelebihan,Kekurangan,Saran Perbaikan
0,Fina,• Belum tampak dominan,• Menentukan volume bangun ruang sisi datar dari konteks kehidupan sehari-hari • Menentukan volume limas dari dimensi yang diketahui • Menganalisis luas permukaan balok dalam konteks kehidupan sehari-hari • Menganalisis luas permukaan balok dari masalah kontekstual • Mengaplikasikan rumus volume limas segitiga dalam konteks nyata • Menggunakan konsep luas permukaan kubus dalam situasi kontekstual • Menggunakan rumus volume limas dalam konteks nyata • Menghitung luas permukaan kubus dari situasi nyata • Menghitung volume prisma dari permasalahan nyata • Menyelesaikan masalah volume prisma dari situasi kehidupan sehari-hari,• Perbanyak latihan pada indikator: menentukan volume bangun ruang sisi datar dari konteks kehidupan sehari-hari. • Perbanyak latihan pada indikator: menentukan volume limas dari dimensi yang diketahui. • Perbanyak latihan pada indikator: menganalisis luas permukaan balok dalam konteks kehidupan sehari-hari. • Perbanyak latihan pada indikator: menganalisis luas permukaan balok dari masalah kontekstual. • Perbanyak latihan pada indikator: mengaplikasikan rumus volume limas segitiga dalam konteks nyata. • Perbanyak latihan pada indikator: menggunakan konsep luas permukaan kubus dalam situasi kontekstual. • Perbanyak latihan pada indikator: menggunakan rumus volume limas dalam konteks nyata. • Perbanyak latihan pada indikator: menghitung luas permukaan kubus dari situasi nyata. • Perbanyak latihan pada indikator: menghitung volume prisma dari permasalahan nyata. • Perbanyak latihan pada indikator: menyelesaikan masalah volume prisma dari situasi kehidupan sehari-hari.
1,fino,• Menentukan volume limas dari dimensi yang diketahui • Menggunakan rumus volume limas dalam konteks nyata • Menghitung volume prisma dari permasalahan nyata,• Menentukan volume bangun ruang sisi datar dari konteks kehidupan sehari-hari • Menganalisis luas permukaan balok dalam konteks kehidupan sehari-hari • Menganalisis luas permukaan balok dari masalah kontekstual • Mengaplikasikan rumus volume limas segitiga dalam konteks nyata • Menggunakan konsep luas permukaan kubus dalam situasi kontekstual • Menghitung luas permukaan kubus dari situasi nyata • Menyelesaikan masalah volume prisma dari situasi kehidupan sehari-hari,• Perbanyak latihan pada indikator: menentukan volume bangun ruang sisi datar dari konteks kehidupan sehari-hari. • Perbanyak latihan pada indikator: menganalisis luas permukaan balok dalam konteks kehidupan sehari-hari. • Perbanyak latihan pada indikator: menganalisis luas permukaan balok dari masalah kontekstual. • Perbanyak latihan pada indikator: mengaplikasikan rumus volume limas segitiga dalam konteks nyata. • Perbanyak latihan pada indikator: menggunakan konsep luas permukaan kubus dalam situasi kontekstual. • Perbanyak latihan pada indikator: menghitung luas permukaan kubus dari situasi nyata. • Perbanyak latihan pada indikator: menyelesaikan masalah volume prisma dari situasi kehidupan sehari-hari.
2,joko,• Menentukan volume limas dari dimensi yang diketahui • Menggunakan rumus volume limas dalam konteks nyata,• Menentukan volume bangun ruang sisi datar dari konteks kehidupan sehari-hari • Menganalisis luas permukaan balok dalam konteks kehidupan sehari-hari • Menganalisis luas permukaan balok dari masalah kontekstual • Mengaplikasikan rumus volume limas segitiga dalam konteks nyata • Menggunakan konsep luas permukaan kubus dalam situasi kontekstual • Menghitung luas permukaan kubus dari situasi nyata • Menghitung volume prisma dari permasalahan nyata • Menyelesaikan masalah volume prisma dari situasi kehidupan sehari-hari,• Perbanyak latihan pada indikator: menentukan volume bangun ruang sisi datar dari konteks kehidupan sehari-hari. • Perbanyak latihan pada indikator: menganalisis luas permukaan balok dalam konteks kehidupan sehari-hari. • Perbanyak latihan pada indikator: menganalisis luas permukaan balok dari masalah kontekstual. • Perbany


📁 File 'hasil_analisis_dengan_saran.xlsx' telah dibuat dan siap diunduh.
